# EEG Preprocessing — Brain Invaders (bi2015a)

## 1. Imports

In [1]:
%matplotlib inline                   
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne         
from pathlib import Path

mne.set_log_level('WARNING')  # suppress MNE's verbose progress messages; change to 'INFO' to debug

/Users/florijnvzuilen/opt/anaconda3/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## 2. Configuration

**Change SUBJECT and SESSION here before running the notebook.** All file paths and labels are built from these two numbers automatically — you never need to edit a path manually.

| Variable | What it controls |
|----------|-----------------|
| SUBJECT | Which participant's data to load (integers 1–43). Subjects 1 and 27 are flagged as bad recordings in the bi2015a paper — avoid them. |
| SESSION | Which recording session: 1 (slow flashes, 110 ms), 2 (medium, 80 ms), or 3 (fast, 50 ms) |

In [2]:
# USER CONFIG 
SUBJECT     = 2   # participant number (1–43); avoid 1 and 27 (bad recordings)
SESSION     = 1   # recording session: 1 (slow flashes), 2 (medium), 3 (fast)

_PROJECT_ROOT = Path().resolve()
DATA_ROOT   = _PROJECT_ROOT / "data" / "raw" / f"subject_{SUBJECT:02d}_csv"
OUTPUT_ROOT = _PROJECT_ROOT / "data" / "preprocessed"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Flash duration per session (ms) — from Table 2 of the bi2015a paper.
# Slower flashes give the brain more time to respond; useful context when
# comparing classifier performance across sessions.
FLASH_DURATION_MS = {
    1: 110,   # slowest: 110 ms on-screen
    2: 80,
    3: 50,    # fastest: 50 ms on-screen
}

# DERIVED 
csv_path = DATA_ROOT   / f"subject_{SUBJECT:02d}_session_{SESSION:02d}.csv"
fif_path = OUTPUT_ROOT / f"subject_{SUBJECT:02d}_session_{SESSION:02d}_epo.fif"
flash_ms = FLASH_DURATION_MS[SESSION]
print(f"Subject  : {SUBJECT}")
print(f"Session  : {SESSION}  (flash duration = {flash_ms} ms)")
print(f"Input    : {csv_path}")
print(f"Output   : {fif_path}")
print(f"Flash duration : {flash_ms} ms  (Session 1=110ms, Session 2=80ms, Session 3=50ms)")

Subject  : 2
Session  : 1  (flash duration = 110 ms)
Input    : /Users/florijnvzuilen/Downloads/project test/data/raw/subject_02_csv/subject_02_session_01.csv
Output   : /Users/florijnvzuilen/Downloads/project test/data/preprocessed/subject_02_session_01_epo.fif
Flash duration : 110 ms  (Session 1=110ms, Session 2=80ms, Session 3=50ms)


## 3. Load the CSV

The bi2015a CSV files have **no header row** — the first row is already data. We supply the 35 column names manually based on the dataset documentation.

The Trigger and Target columns are stored as floats (0.0 / 1.0) in some sessions due to how the files were written; we cast them to int for clarity.

In [3]:
COLUMNS = [
    "Time",
    "FP1", "FP2", "AFz", "F7",  "F3",  "F4",  "F8",
    "FC5", "FC1", "FC2", "FC6", "T7",  "C3",  "Cz",  "C4",  "T8",
    "CP5", "CP1", "CP2", "CP6", "P7",  "P3",  "Pz",  "P4",  "P8",
    "PO7", "O1",  "Oz",  "O2",  "PO8", "PO9", "PO10",
    "Trigger", "Target",
]

EEG_CHANNELS = COLUMNS[1:33]   
SFREQ        = 512.0 

df = pd.read_csv(csv_path, header=None, names=COLUMNS)
df["Trigger"] = df["Trigger"].astype(int)
df["Target"]  = df["Target"].astype(int)

print(f"Shape    : {df.shape[0]:,} samples × {df.shape[1]} columns")
print(f"Duration : {df['Time'].max():.1f} s  ({df['Time'].max()/60:.1f} min)")
print(f"Trigger=1 events : {(df['Trigger']==1).sum()}")
print(f"Target=1  events : {(df['Target']==1).sum()}")
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/florijnvzuilen/Downloads/project test/data/raw/subject_02_csv/subject_02_session_01.csv'

## 4. Build an MNE RawArray

In [ ]:
# EEG data: select the 32 electrode columns, transpose to (channels, samples),
# and convert µV → V by dividing by 1,000,000 — MNE works in Volts internally
eeg_data  = df[EEG_CHANNELS].values.T / 1e6

stim_data = df["Trigger"].values[np.newaxis, :]

raw_data  = np.vstack([eeg_data, stim_data])

ch_names = EEG_CHANNELS + ["STI014"]
ch_types = ["eeg"] * len(EEG_CHANNELS) + ["stim"]
info = mne.create_info(ch_names=ch_names, sfreq=SFREQ, ch_types=ch_types)
raw  = mne.io.RawArray(raw_data, info, verbose=False)

print(raw)

## 5. Montage

In [ ]:
montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage, match_case=False, on_missing="warn")
print("Montage applied.")

## 6. Visualisation: Raw PSD

Plot the power spectral density of the **unfiltered** raw signal. What to look for:

- **50 Hz peak** — a sharp spike at 50 Hz indicates European power-line interference; it should disappear after the notch filter
- **1/f shape** — EEG power naturally decreases with frequency. A heavily distorted slope or flat spectrum can indicate amplifier saturation or bad channels
- **Channel spread** — `average=False` plots each channel individually so outliers are visible

In [ ]:
spectrum = raw.compute_psd(fmax=60)   # compute power at each frequency from 0 to 60 Hz

psds_db  = 10 * np.log10(spectrum.get_data() * 1e12)  # V²/Hz → dB µV²/Hz

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(spectrum.freqs, psds_db.T, alpha=0.35, color='steelblue', linewidth=0.5)
ax.plot(spectrum.freqs, psds_db.mean(axis=0), color='black', linewidth=2)  # channel-average in bold
ax.axvline(50, color='red', linestyle='--', alpha=0.8, label='50 Hz (power line)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB µV²/Hz)')
ax.set_title('PSD — Raw signal (each line = 1 channel)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Filtering

Two filters are applied in sequence:

1. **Notch filter at 50 Hz** — removes power-line interference (European electrical grid).
2. **Bandpass 0.1–30 Hz** — removes slow drift (below 0.1 Hz) and high-frequency noise above 30 Hz. The P300 component lives in the 1–10 Hz range, so nothing physiologically relevant is lost.

In [ ]:
raw_notched = raw.copy().notch_filter(freqs=50, picks="eeg")
print("Notch filter at 50 Hz applied.")

### Visualisation: PSD after Notch Filter

The **50 Hz peak should now be gone**. Compare this plot to the raw PSD above — everything else should look identical. If the peak is still visible, the notch filter may not have been applied to all channels.

In [ ]:
spectrum = raw_notched.compute_psd(fmax=60)   # recompute PSD on the notch-filtered signal
psds_db  = 10 * np.log10(spectrum.get_data() * 1e12)  # V²/Hz → dB µV²/Hz (same conversion as before)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(spectrum.freqs, psds_db.T, alpha=0.35, color='steelblue', linewidth=0.5)
ax.plot(spectrum.freqs, psds_db.mean(axis=0), color='black', linewidth=2)
ax.axvline(50, color='red', linestyle='--', alpha=0.8, label='50 Hz (should be flat now)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB µV²/Hz)')
ax.set_title('PSD — After notch filter at 50 Hz')
ax.legend()
plt.tight_layout()
plt.show()

### Step 2: Bandpass filter (0.1–30 Hz)

The notch filter only removed the narrow 50 Hz spike. We now apply a **bandpass filter** to cut everything outside the 0.1–30 Hz range.

- **Low-cut at 0.1 Hz (high-pass)** — removes very slow voltage drifts caused by electrode sweat, body movement, and DC offsets. These drifts can be tens of µV and would corrupt the baseline correction we apply during epoching.
- **High-cut at 30 Hz (low-pass)** — removes high-frequency muscle noise (EMG) from jaw clenching or scalp tension, and any residual electrical noise. Brain signals measured at the scalp for P300-based BCIs carry essentially no useful information above 30 Hz.

The P300 component peaks around 300 ms post-stimulus and is dominated by frequencies between 1–10 Hz — well within the 0.1–30 Hz passband, so nothing physiologically relevant is discarded.

In [ ]:
raw_filtered = raw_notched.copy().filter(l_freq=0.1, h_freq=30, picks="eeg")
# l_freq=0.1: high-pass edge — removes anything below 0.1 Hz (slow electrode drift)
# h_freq=30:  low-pass edge  — removes anything above 30 Hz (muscle noise, EMG)
print("Bandpass filter 0.1–30 Hz applied.")

### Visualisation: PSD after Bandpass Filter

The signal should now **drop sharply below 0.1 Hz and above 30 Hz**. The retained 0.1–30 Hz band contains all P300-relevant activity — the N200, P300 and late positive components all live below 10 Hz.

In [ ]:
spectrum = raw_filtered.compute_psd(fmax=60)   # recompute PSD on the fully filtered signal
psds_db  = 10 * np.log10(spectrum.get_data() * 1e12)  # V²/Hz → dB µV²/Hz

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(spectrum.freqs, psds_db.T, alpha=0.35, color='steelblue', linewidth=0.5)
ax.plot(spectrum.freqs, psds_db.mean(axis=0), color='black', linewidth=2)
ax.axvspan(0, 0.1, alpha=0.12, color='red', label='Filtered out (< 0.1 Hz)')   # axvspan = shaded vertical band
ax.axvspan(30, 60, alpha=0.12, color='red', label='Filtered out (> 30 Hz)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB µV²/Hz)')
ax.set_title('PSD — After bandpass filter (0.1–30 Hz)')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Event detection & Epoching

MNE scans the STIM channel for rising edges to locate stimulus onsets. Each detected event becomes the centre of a 1.2-second epoch:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `tmin` | −0.2 s | 200 ms pre-stimulus baseline |
| `tmax` | 1.0 s | captures P300 (300–600 ms) and late slow-wave components up to 1 s |
| Baseline | (−0.2, 0) | subtract mean of pre-stimulus window |
| Reject threshold | 100 µV peak-to-peak | simple artefact rejection; no ICA |

In [ ]:
events = mne.find_events(raw_filtered, stim_channel="STI014", verbose=False)
print(f"Events found : {len(events)}")
print(f"Event IDs    : {np.unique(events[:, 2])}") 

### Visualisation: Event Timing

Plot stimulus onsets over time. What to check:

- **Event count** should match `Trigger=1` from Section 3
- **Even spacing** — events should be distributed throughout the recording with no large unexplained gaps (gaps may indicate dropped triggers or a pause in the experiment)

In [ ]:
mne.viz.plot_events(
    events,
    sfreq=raw_filtered.info['sfreq'],
    first_samp=raw_filtered.first_samp,
)

### Cutting the continuous signal into epochs

**Key parameters explained:**

| Parameter | Value | What it does |
|-----------|-------|--------------|
| tmin=-0.2 | −200 ms | Epoch starts 200 ms *before* the flash — this is the **pre-stimulus baseline** period |
| tmax=1.0 | +1000 ms | Epoch ends 1000 ms *after* the flash — enough to see the P300 peak (~300–500 ms) and later components |
| baseline=(-0.2, 0) | pre-stimulus window | **Baseline correction**: subtracts the mean voltage during −200→0 ms from every timepoint. This removes any slow voltage offset that happened to be present before the flash, so all epochs start from a common zero and trial-to-trial differences reflect only brain *responses* |
| reject={"eeg": 100e-6} | 100 µV | **Artefact rejection**: discards any epoch in which *any* channel exceeds ±100 µV. Genuine brain signals are 5–20 µV; eye blinks produce 100–300 µV spikes. This threshold removes the most contaminated trials without ICA. `100e-6` is 100 µV expressed in Volts (MNE's internal unit) |
| preload=True | — | Loads all epoch data into RAM immediately; required for attaching metadata and saving |

In [ ]:
epochs = mne.Epochs(
    raw_filtered,
    events,
    event_id={"stimulus": 1},
    tmin=-0.2,
    tmax=1.0,
    baseline=(-0.2, 0),
    reject={"eeg": 100e-6},     # discard epochs where any EEG channel exceeds ±100 µV
    preload=True, 
    verbose=False,
)

print(epochs)

## 9. Extract labels

The Target column records whether each stimulus was a target (1) or non-target (0). We align these labels to the trigger sample indices returned by mne.find_events, then keep only the labels for epochs that survived artefact rejection.

In [ ]:
# events[:, 0] contains the sample index of each flash in the raw recording.
# Subtracting raw_filtered.first_samp converts MNE's internal sample index
# into a row index into our original DataFrame, so we can look up the Target label.
labels      = df["Target"].values[events[:, 0] - raw_filtered.first_samp]

labels_kept = labels[epochs.selection]

n_target    = int(labels_kept.sum())    
n_nontarget = len(labels_kept) - n_target

print(f"Epochs after rejection : {len(labels_kept)}  "
      f"(dropped {len(events) - len(labels_kept)})")
print(f"  Target     (1) : {n_target}")
print(f"  Non-target (0) : {n_nontarget}")
if n_target > 0:
    print(f"  Ratio          : 1 : {n_nontarget / n_target:.1f}")

### Attaching labels to the epoch object

We now have two separate things that belong together:
1. The cleaned EEG data for each surviving epoch (inside the epochs object)
2. A target / non-target label for each epoch (labels_kept)

In [ ]:
epochs.metadata = pd.DataFrame(
    {"target": labels_kept},
    index=range(len(labels_kept)),
)

print(epochs.metadata["target"].value_counts()
      .rename(index={0: "non-target", 1: "target"})
      .to_string())

### Visualisation: ERP at Cz — Target vs Non-target

This is the **key sanity check** for a P300 dataset. The target ERP should show a clear positive peak (**P300**) between 300–600 ms post-stimulus that the non-target ERP does not. If both curves are flat and indistinguishable, it likely indicates a label alignment problem or a noisy recording.

In [ ]:
# Filter the epoch object by metadata label
target_epochs    = epochs[epochs.metadata['target'] == 1]
nontarget_epochs = epochs[epochs.metadata['target'] == 0]

fig, ax = plt.subplots(figsize=(8, 4))

# Average all target epochs at channel Cz.
# Individual trials are too noisy to see the ERP; averaging hundreds of trials
# cancels out random neural activity, leaving only the time-locked P300 response.
ax.plot(epochs.times,
        target_epochs.get_data(picks=['Cz']).mean(axis=0).squeeze() * 1e6,
        label='Target', color='red')
ax.plot(epochs.times,
        nontarget_epochs.get_data(picks=['Cz']).mean(axis=0).squeeze() * 1e6,
        label='Non-target', color='blue')
ax.axvline(0, color='black', linestyle='--', label='Stimulus onset')
ax.axhline(0, color='gray', linestyle=':')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (µV)')
ax.set_title(f'ERP at Cz — Subject {SUBJECT}, Session {SESSION} ({flash_ms} ms flash)')
ax.legend()
plt.tight_layout()
plt.show()

### Visualisation: Topographic Map of the P300

Scalp topography of the **target-average ERP** at 300, 400, and 500 ms. The P300 has a **central-parietal positive distribution** — expect a warm-coloured blob strongest over Pz, Cz, and CPz. A frontal or occipital distribution is atypical and warrants inspection.

In [ ]:
target_epochs.average().plot_topomap(times=[0.3, 0.4, 0.5], average=0.05)

## 10. Save epochs

Epochs are saved as a .fif file (MNE's native format). This file can be loaded directly in subsequent analysis notebooks with mne.read_epochs().

In [ ]:
epochs.save(fif_path, overwrite=True)   # overwrite=True: safe to re-run — replaces any existing file for this subject/session
print(f"Saved: {fif_path}")